## Download a single DTM + DSM tile

Independent of the merge above: just one BEV tile, for both products, covering a
point of interest. `fetch_bev_als.point_to_tile_id()` maps the point to its
containing tile (doesn't handle points near a tile border - just picks the single
containing tile, which is fine here).

In [ ]:
from fetch_bev_als import download_tiles, point_to_tile_id

# Großglocker
GG_LAT = 47.0742
GG_LON = 12.6938
# Stephansdom
GG_LAT = 48.208492
GG_LON = 16.373127
ZOOM = 17

GG_TILE_ID = point_to_tile_id(GG_LAT, GG_LON)
DTM_DIR = "../data/als_tiles/DTM"
DSM_DIR = "../data/als_tiles/DSM"

dtm_tile_path = download_tiles("DTM", [GG_TILE_ID], DTM_DIR)[0]
dsm_tile_path = download_tiles("DSM", [GG_TILE_ID], DSM_DIR)[0]
dtm_tile_path, dsm_tile_path

## Evaluate Tile Bounds

In [ ]:
import math

import rasterio.warp


def deg2tile(lat: float, lon: float, z: int) -> tuple[int, int]:
    """WGS84 lon/lat -> XYZ (Google/OSM scheme) tile indices at zoom z."""
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    lat_rad = math.radians(lat)
    y = int((1.0 - math.log(math.tan(lat_rad) + 1.0 / math.cos(lat_rad)) / math.pi) / 2.0 * n)
    return x, y


def tile_bounds_lonlat(x: int, y: int, z: int) -> tuple[float, float, float, float]:
    """XYZ tile indices -> (west, south, east, north) in WGS84 degrees."""
    n = 2 ** z

    def lat_of_row(row):
        return math.degrees(math.atan(math.sinh(math.pi * (1 - 2 * row / n))))

    west = x / n * 360.0 - 180.0
    east = (x + 1) / n * 360.0 - 180.0
    north = lat_of_row(y)
    south = lat_of_row(y + 1)
    return west, south, east, north


tile_x, tile_y = deg2tile(GG_LAT, GG_LON, ZOOM)
bounds_lonlat = tile_bounds_lonlat(tile_x, tile_y, ZOOM)
bounds_3857 = rasterio.warp.transform_bounds("EPSG:4326", "EPSG:3857", *bounds_lonlat)

print(f"tile z={ZOOM} x={tile_x} y={tile_y}")
print(f"bounds (lon/lat): {bounds_lonlat}")
print(f"bounds (EPSG:3857): {bounds_3857}")

## Reproject into the 256x256 Web Mercator grid

In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.warp import reproject

TILE_SIZE = 256
dst_transform = from_bounds(*bounds_3857, TILE_SIZE, TILE_SIZE)


def reproject_to_tile(src_path: str) -> np.ndarray:
    with rasterio.open(src_path) as src:
        dst = np.zeros((TILE_SIZE, TILE_SIZE), dtype=np.float32)
        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dst_transform,
            dst_crs="EPSG:3857",
            resampling=Resampling.bilinear,
        )
        return dst


dtm_arr = reproject_to_tile(dtm_tile_path)
dsm_arr = reproject_to_tile(dsm_tile_path)
dtm_arr.shape, dsm_arr.shape, dtm_arr.min(), dtm_arr.max()

## Web Mercator ground-scale correction (altitude vs. quad size)

Web Mercator inflates horizontal distances away from the equator by `1/cos(lat)`.
weBIGeo's renderer corrects for this by scaling the *height* sample instead of the
horizontal quad size:

```wgsl
let world_space_y: f32 = (*position).y + camera.position.y;
let altitude_correction_factor: f32 = 0.125 / cos(y_to_lat(world_space_y));  // AlpineMapsOrg/renderer#5
let adjusted_altitude: f32 = altitude_tex * altitude_correction_factor;
```

Mathematically this is equivalent to what this notebook did before (scaling
`quad_width`/`quad_height` by `cos(lat)`) — either way the height/distance ratio
that drives the normal direction ends up the same. To match weBIGeo exactly,
`quad_width`/`quad_height` below are now the *raw*, uncorrected Web Mercator
meters/pixel, and the correction is applied to the height values instead via
`altitude_correction_factor = 1 / cos(lat)` (their `0.125` is a raw-texture-value
-to-meters unit scale that doesn't apply here since `dtm_arr`/`dsm_arr` are
already float meters).

`APPLY_ALTITUDE_CORRECTION` toggles it on/off — flip it and re-run the cells below
to compare `dtm_h`/`dsm_h` (used everywhere downstream instead of the raw
`dtm_arr`/`dsm_arr`) with and without the correction.

In [ ]:
APPLY_ALTITUDE_CORRECTION = True  # flip and re-run downstream cells to compare

WEB_MERCATOR_RES_ZOOM0 = 156543.03392804097  # meters/pixel at zoom 0, equator
quad_width = quad_height = WEB_MERCATOR_RES_ZOOM0 / 2 ** ZOOM  # raw, uncorrected

# weBIGeo folds the Web Mercator latitude distortion into the height value
# instead of the horizontal quad size - see AlpineMapsOrg/renderer#5. Their 0.125
# factor also converts a raw quantized height-texture value to meters, which
# doesn't apply here since dtm_arr/dsm_arr are already float meters.
altitude_correction_factor = 1.0 / math.cos(math.radians(GG_LAT))


def corrected_height(height: np.ndarray) -> np.ndarray:
    return height * altitude_correction_factor if APPLY_ALTITUDE_CORRECTION else height


dtm_h = corrected_height(dtm_arr)
dsm_h = corrected_height(dsm_arr)

print(f"quad_width = quad_height = {quad_width:.4f} m (raw Web Mercator, uncorrected)")
print(f"altitude_correction_factor = {altitude_correction_factor:.4f} (applied={APPLY_ALTITUDE_CORRECTION})")

## Fetch the matching basemap.at orthophoto tile

Same `z/x/y` tile, real orthophoto imagery from basemap.at (same source/URL scheme as `tile_creators/debug_ortho.py`). A visual reference to sanity-check the normal maps against actual terrain features (ridges, rock, snow/ice) visible in the photo.

In [ ]:
import io
import os

import requests
from PIL import Image

BASEMAP_URL_TEMPLATE = "https://gataki.cg.tuwien.ac.at/raw/basemap/tiles/{z}/{y}/{x}.jpeg"
ORTHO_DIR = "../cache/tmp_normal_test/ortho"


def fetch_basemap_tile(z: int, x: int, y: int) -> Image.Image:
    """Download the tile (skipping if already cached on disk) and return it as an image."""
    os.makedirs(ORTHO_DIR, exist_ok=True)
    path = os.path.join(ORTHO_DIR, f"{z}_{x}_{y}.jpeg")
    if os.path.exists(path):
        return Image.open(path).convert("RGB")
    url = BASEMAP_URL_TEMPLATE.format(z=z, y=y, x=x)
    r = requests.get(url, timeout=15)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)
    return Image.open(io.BytesIO(r.content)).convert("RGB")


ortho_img = fetch_basemap_tile(ZOOM, tile_x, tile_y)
ortho_img

## Normal map: weBIGeo finite-difference method

Direct port of `normal_by_finite_difference_method` from weBIGeo's `normal.wgsl`
shader (the exact method the renderer uses at draw time). The shader's own
`altitude_correction_factor` (see the cell above) is applied upstream to produce
`dtm_h`/`dsm_h`, so it's implicitly `1.0` by the time it reaches this function - no
border half-width correction either (edge texels just clamp — tile-border
correctness doesn't matter for this test).

This is a plain 4-neighbor (axis-aligned L/R/U/D only) central difference, *not* the
classic 3x3 Sobel kernel below — cheaper, no diagonal-neighbor blending.

Note ported as-is from the shader: its `x` and `y` components use opposite sign
conventions (`nx = hL - hR`, but `ny = hD - hU` with no swap) — a known quirk of the
stackoverflow-derived formula the shader itself references. It's preserved here
rather than "fixed", and the Sobel comparison below matches this same asymmetry so
the two methods are only compared on kernel support, not confounded by a differing
sign convention.

In [ ]:
def normal_by_finite_difference(height: np.ndarray, quad_width: float, quad_height: float) -> np.ndarray:
    padded = np.pad(height, 1, mode="edge")
    hL = padded[1:-1, :-2]
    hR = padded[1:-1, 2:]
    hD = padded[2:, 1:-1]
    hU = padded[:-2, 1:-1]

    nx = (hL - hR) / quad_width
    ny = (hD - hU) / quad_height
    nz = np.full_like(nx, 2.0)

    normal = np.stack([nx, ny, nz], axis=-1)
    normal /= np.linalg.norm(normal, axis=-1, keepdims=True)
    return normal


def normal_to_rgb(normal: np.ndarray) -> np.ndarray:
    return ((normal * 0.5 + 0.5) * 255).clip(0, 255).astype(np.uint8)


dtm_normal_fd_rgb = normal_to_rgb(normal_by_finite_difference(dtm_h, quad_width, quad_height))
dsm_normal_fd_rgb = normal_to_rgb(normal_by_finite_difference(dsm_h, quad_width, quad_height))

## Normal map: standard Sobel operator (for comparison)

Classic 3x3 Sobel kernels (blends the 4 diagonal neighbors too, unlike the
finite-difference method above), same real-world `quad_width`/`quad_height` so the
two are on the same scale. The x-axis uses the standard `-dz/dx` negation, but the
y-axis is intentionally left un-negated to match weBIGeo's own asymmetric
convention (see note above) — otherwise the comparison would be confounded by two
differing sign conventions instead of just kernel support. No scipy dependency
needed (the `clouds` conda env doesn't have it) — this is a plain numpy/`np.pad`
implementation.

In [ ]:
def normal_by_sobel(height: np.ndarray, quad_width: float, quad_height: float) -> np.ndarray:
    padded = np.pad(height, 1, mode="edge")
    tl, tm, tr = padded[:-2, :-2], padded[:-2, 1:-1], padded[:-2, 2:]
    ml, mr = padded[1:-1, :-2], padded[1:-1, 2:]
    bl, bm, br = padded[2:, :-2], padded[2:, 1:-1], padded[2:, 2:]

    gx = -(tl + 2 * ml + bl) + (tr + 2 * mr + br)
    gy = -(tl + 2 * tm + tr) + (bl + 2 * bm + br)

    dzdx = (gx / 8.0) / quad_width
    dzdy = (gy / 8.0) / quad_height

    # y intentionally not negated (unlike the standard -dz/dy): the weBIGeo
    # shader's own x/y convention is asymmetric (nx=hL-hR, but ny=hD-hU with
    # no swap), so matching that here keeps this a fair comparison of kernel
    # support (4-neighbor vs 3x3+diagonals) rather than differing sign
    # conventions.
    normal = np.stack([-dzdx, dzdy, np.ones_like(dzdx)], axis=-1)
    normal /= np.linalg.norm(normal, axis=-1, keepdims=True)
    return normal


dtm_normal_sobel_rgb = normal_to_rgb(normal_by_sobel(dtm_h, quad_width, quad_height))
dsm_normal_sobel_rgb = normal_to_rgb(normal_by_sobel(dsm_h, quad_width, quad_height))

## Visualize

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 4, figsize=(15, 8))
rows = [
    ("DTM", dtm_h, dtm_normal_fd_rgb, dtm_normal_sobel_rgb),
    ("DSM", dsm_h, dsm_normal_fd_rgb, dsm_normal_sobel_rgb),
]
for row, (label, height, fd_rgb, sobel_rgb) in enumerate(rows):
    axes[row, 0].imshow(ortho_img)
    axes[row, 0].set_title("basemap.at orthophoto")
    axes[row, 1].imshow(height, cmap="gray")
    axes[row, 1].set_title(f"{label} height")
    axes[row, 2].imshow(fd_rgb)
    axes[row, 2].set_title(f"{label} normal (weBIGeo finite-diff)")
    axes[row, 3].imshow(sobel_rgb)
    axes[row, 3].set_title(f"{label} normal (Sobel)")
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle(f"z={ZOOM} x={tile_x} y={tile_y} — Großglockner test tile")
plt.tight_layout()
plt.show()

## 3D check: drape the orthophoto over the DSM (interactive)

Extrudes the DSM heightfield and textures it with the basemap.at orthophoto,
rotatable/zoomable via Plotly (`pip install plotly` if not already in this
kernel's env). Since both arrays are already resampled onto the exact same
256x256 grid, any misalignment between the DTM/DSM reprojection and the ortho
fetch (wrong bounds, swapped axes, off-by-one tile index, etc.) would show up
here as image features (ridgelines, buildings, rock/snow boundaries) not lining
up with the 3D bumps.

Plotly's `Surface` only maps a single scalar field through one colorscale (no
true RGB texture support like matplotlib's `facecolors`), so the ortho image is
quantized to a 256-color palette (`Image.quantize`) and that palette becomes a
custom colorscale — a lossy approximation, but close enough to still recognize
features by color/shape. `X`/`Y` use the same latitude-corrected
`quad_width`/`quad_height` as the normal maps above, so the plot is close to
true-to-scale (`Z_EXAGGERATION` is purely cosmetic and doesn't affect alignment).

In [ ]:
import plotly.graph_objects as go

Z_EXAGGERATION = 1.0  # cosmetic only - doesn't affect X/Y alignment

rows = np.arange(TILE_SIZE)
cols = np.arange(TILE_SIZE)
xs = cols * quad_width
ys = (TILE_SIZE - 1 - rows) * quad_height  # flip so north is "up"
Z = dsm_h * Z_EXAGGERATION

ortho_quantized = ortho_img.resize((TILE_SIZE, TILE_SIZE)).quantize(colors=256)
palette = ortho_quantized.getpalette()[:256 * 3]
colorscale = [
    [i / 255, f"rgb({palette[i * 3]},{palette[i * 3 + 1]},{palette[i * 3 + 2]})"]
    for i in range(256)
]
surfacecolor = np.array(ortho_quantized, dtype=np.float64)

fig = go.Figure(data=[go.Surface(
    x=xs, y=ys, z=Z,
    surfacecolor=surfacecolor,
    colorscale=colorscale,
    cmin=0, cmax=255,
    showscale=False,
)])
fig.update_layout(
    title="DSM extruded, textured with the basemap.at orthophoto (256-color quantized)",
    scene=dict(aspectmode="manual", aspectratio=dict(x=1, y=1, z=1)),
    width=800, height=800,
)
fig.show()